In [0]:
%run "../libs/libs"

In [0]:
%run "../common/define_path"

In [0]:
otracking_path

In [0]:
def read_bronze_invoice():
    print('Reading the Bronze Table Data : ',end='')
    df_bronzeinvoice = (spark.readStream
                    .table(f"otracking.bronze.invoice")
                    )
    print(f'Reading otracking.bronze.invoice Success!')
    return df_bronzeinvoice

In [0]:
df_invoice = read_bronze_invoice()

In [0]:
from pyspark.sql.functions import col, to_date
df_invoice = df_invoice.select(col("invno").alias("invoiceno"),
                               col("invdate").alias("invoicedate"),
                               col("ord").alias("orderno"),
                               col("amount").alias("invoiceamount"),
                               col("taxrate"),
                               col("taxamount"),
                               col("paymentdays"),
                               col("nameofcustomer"),
                               col("city").alias("invoicecity"),
                               col("state").alias("invoicestate"))

                               

In [0]:
df_invoice = convert_to_int(df_invoice, "paymentdays")
df_invoice = df_invoice.withColumn("ingestion_time",current_timestamp())

In [0]:
target_columns = ["invoiceamount", "taxrate","taxamount"]

# Apply the UDF in a loop
for col_name in target_columns:
     df_invoice = convert_to_float(df_invoice, col_name)


In [0]:
df_invoice = convert_to_timestamp(df_invoice, "invoicedate")

In [0]:
from pyspark.sql.functions import to_timestamp, col
df_invoice= df_invoice.withColumn("invoicedate", to_timestamp(col("invoicedate")))


In [0]:
def write_silver_invoice(StreamingDF):
      write_StreamSilver = (StreamingDF.writeStream
                .format('delta')
                .option('checkpointLocation',f'{otracking_path}/silver/checkpoint/invoice/')
                .outputMode('append')
                .queryName("silverinvoicestream")
                .trigger(availableNow=True)
                .toTable("otracking.silver.invoice")
            )
      write_StreamSilver.awaitTermination()

In [0]:
write_silver_invoice(df_invoice)

In [0]:
%sql
select * from otracking.silver.invoice limit 1000